# 08 — Memory forecast: how much will still be remembered on exam day?

The progress page forecasts, per notebook, how many studied cards a student will still recall on the day of their
test if they stop reviewing, and which cards to review first. That needs a model of forgetting.

**Model.** Half-life regression (HLR; Settles & Meeder 2016, ACL): recall decays as `p = 2^(-Δ/h)` and the half-life
grows with what the learner got right and shrinks with what they got wrong, `h = 2^(w·x)`,
`x = (√(1+right), √(1+wrong), 1)`. Implemented in `ml_pipeline/memory_model.py`. The lexeme features of the paper are
left out: a model of Spanish words says nothing about a student's own flashcards, while how history shapes
forgetting carries over.

**Data.** Duolingo's public learning traces (12.9M rows; CC BY-NC 4.0; see `datasets/README.md`). Each row is one
practice session of one word: `delta` since it was last seen, the learner's history with it, and `p_recall`, the share
of the session's attempts recalled. A sample of users is taken, then split **by user** (90/10) so no learner's
history is seen in both training and test.

In [1]:
import gzip, hashlib, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from ml_pipeline.notebook_setup import bootstrap
bootstrap()
import numpy as np, pandas as pd
from ml_pipeline import config
from ml_pipeline.memory_model import MemoryModel, fit, recall_probability, half_life

RAW = config.REPO_ROOT / 'datasets' / 'raw' / 'Duolingo' / 'settles.acl16.learning_traces.13m.csv.gz'
cols = ['p_recall', 'delta', 'user_id', 'history_seen', 'history_correct']
keep = []
for chunk in pd.read_csv(RAW, usecols=cols, chunksize=1_000_000):
    # 1 user in 40, chosen by a hash so the sample is the same on every run.
    pick = chunk.user_id.map(lambda u: int(hashlib.md5(u.encode()).hexdigest(), 16) % 40 == 0)
    keep.append(chunk[pick])
df = pd.concat(keep, ignore_index=True)
df['delta_days'] = df.delta / 86400
df['right'] = df.history_correct
df['wrong'] = df.history_seen - df.history_correct
print(f'{len(df):,} sessions from {df.user_id.nunique():,} users; mean p_recall {df.p_recall.mean():.3f}')

356,572 sessions from 2,938 users; mean p_recall 0.896


In [2]:
users = df.user_id.unique()
rng = np.random.default_rng(config.RANDOM_SEED)
test_users = set(rng.choice(users, size=len(users) // 10, replace=False))
test = df[df.user_id.isin(test_users)]
train = df[~df.user_id.isin(test_users)]
print(f'train {len(train):,} rows / test {len(test):,} rows')

train 321,007 rows / test 35,565 rows


## Models

In [3]:
from scipy.optimize import minimize_scalar
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def rows(frame):
    return zip(frame.delta_days, frame.right, frame.wrong, frame.p_recall)

# 1. Constant: the training mean, whatever the history.
const = train.p_recall.mean()

# 2. One half-life for everything: forgetting over time, no history.
def one_h_loss(log_h):
    return np.mean((2.0 ** (-train.delta_days / 2.0 ** log_h) - train.p_recall) ** 2)
one_h = 2.0 ** minimize_scalar(one_h_loss, bounds=(-6, 8), method='bounded').x

# 3. Logistic regression on the same inputs; p_recall as a soft label
#    (each row twice: recalled with weight p, forgotten with weight 1-p).
def lr_x(frame):
    return np.column_stack([np.log1p(frame.delta_days), np.sqrt(1 + frame.right), np.sqrt(1 + frame.wrong)])
X = lr_x(train)
logit = LogisticRegression(max_iter=1000).fit(
    np.vstack([X, X]), np.r_[np.ones(len(X)), np.zeros(len(X))],
    sample_weight=np.r_[train.p_recall, 1 - train.p_recall])

# 4. Half-life regression, as fitted (no sign limits) and held to the signs
#    a study app needs: a right answer never shortens memory, a wrong one never
#    lengthens it. Penalties off: a validation split of the training users
#    (scratch run) gave MAE 0.122 with both at 0 and 0.168 with the paper's l2.
hlr_free = fit(rows(train), signed=False)
hlr = fit(rows(train))
print('HLR, no limits', {k: round(v, 3) for k, v in hlr_free.weights.items()})
print('HLR, signed    ', {k: round(v, 3) for k, v in hlr.weights.items()})
print('single half-life', round(one_h, 1), 'days')

HLR, no limits {'right': 3.258, 'wrong': 2.128, 'bias': 2.13}
HLR, signed     {'right': 3.909, 'wrong': 0.0, 'bias': 2.874}
single half-life 256.0 days


In [4]:
from sklearn.metrics import log_loss

def evaluate(name, pred):
    y = test.p_recall.to_numpy()
    pred = np.clip(pred, 1e-4, 1 - 1e-4)
    return {'model': name, 'MAE': np.mean(np.abs(pred - y)),
            # Log-loss treats p_recall as the probability it is: it punishes a
            # confident wrong forecast, which MAE barely notices.
            'log-loss': log_loss(np.r_[np.ones(len(y)), np.zeros(len(y))], np.r_[pred, pred],
                                 sample_weight=np.r_[y, 1 - y]),
            'AUC (all recalled vs not)': roc_auc_score((y == 1.0).astype(int), pred)}

def hlr_predict(model):
    return np.array([recall_probability(model, d, r, w) for d, r, w, _ in rows(test)])

results = pd.DataFrame([
    evaluate('always recalled', np.ones(len(test))),
    evaluate('constant (mean)', np.full(len(test), const)),
    evaluate('one half-life', 2.0 ** (-test.delta_days.to_numpy() / one_h)),
    evaluate('logistic regression', logit.predict_proba(lr_x(test))[:, 1]),
    evaluate('HLR, no limits', hlr_predict(hlr_free)),
    evaluate('HLR, signed', hlr_predict(hlr)),
])
results.round(4)

,model,MAE,log-loss,AUC (all recalled vs not)
0,always recalled,0.0942,0.8669,0.5000
1,constant (mean),0.1667,0.3126,0.5000
2,one half-life,0.1067,0.5913,0.5509
3,logistic regression,0.1623,0.3077,0.6040
4,"HLR, no limits",0.1059,0.5943,0.5509
5,"HLR, signed",0.1059,0.5943,0.5509


## What the fitted model says

In [5]:
days = [1, 3, 7, 14, 30]
curves = {f'{r} right, {w} wrong (h = {half_life(hlr, r, w):.1f} d)': [recall_probability(hlr, d, r, w) for d in days]
          for r, w in [(1, 0), (3, 0), (6, 0), (3, 2)]}
pd.DataFrame(curves, index=[f'{d} days' for d in days]).round(2)

,"1 right, 0 wrong (h = 274.0 d)","3 right, 0 wrong (h = 274.0 d)","6 right, 0 wrong (h = 274.0 d)","3 right, 2 wrong (h = 274.0 d)"
1 days,1.00,1.00,1.00,1.00
3 days,0.99,0.99,0.99,0.99
7 days,0.98,0.98,0.98,0.98
14 days,0.97,0.97,0.97,0.97
30 days,0.93,0.93,0.93,0.93


In [6]:
out = config.REPO_ROOT / 'ml_pipeline' / 'models' / 'memory_hlr.json'
hlr.save(out)
print('saved', out.relative_to(config.REPO_ROOT))

saved ml_pipeline/models/memory_hlr.json


## Decision

**The app does not use this model.** It forecasts with FSRS-6 (`ml_pipeline/fsrs_model.py`), the published model
Anki uses, with its default parameters fitted on Anki flashcard logs. This notebook stays as the experiment that
led there.

What the table above shows:

- **MAE rewards saying "remembered" to everyone.** Sixty per cent of sessions have `p_recall = 1`, so the trivial
  "always recalled" forecast has the lowest MAE of all (0.094), lower than any fitted model. The paper's metric
  does not tell good forecasts from lazy ones on this data. Log-loss does, and on log-loss the constant mean (0.313)
  is close to the best model.
- **The paper's HLR degenerates here.** Fitted on squared error, every half-life runs to the 274-day cap: "93%
  recalled after 30 days" for any history. Unconstrained, it also learns that wrong answers lengthen memory,
  because in these traces more wrong answers come with more exposure. Fitting on log-loss instead (scratch run)
  gives the right signs without constraints (3 right, 0 wrong: 88-day half-life; 3 right, 2 wrong: 38 days), but
  its log-loss (0.558) is worse than the constant, since HLR assumes recall is certain just after a review and
  Duolingo learners recall about 90% even then.
- **History carries little signal in this data.** The best model (logistic regression, AUC 0.60) is only a little
  better than the mean. Duolingo practises the same words daily in short sessions; a student's flashcards are
  reviewed days apart and forgotten much faster.

FSRS-6 forgetting curves, for comparison (computed with `ml_pipeline/fsrs_model.py`):

| history | S (days) | 1 d | 3 d | 7 d | 14 d | 30 d |
|---|---|---|---|---|---|---|
| 1 Good | 2.3 | 0.95 | 0.88 | 0.81 | 0.74 | 0.67 |
| 3 Good (days 0, 3, 10) | 39.2 | 1.00 | 0.99 | 0.98 | 0.95 | 0.92 |
| Good, then forgot (day 5) | 0.7 | 0.87 | 0.77 | 0.69 | 0.62 | 0.56 |

A card seen once fades within a week; one reviewed three times holds for a month; a forgotten one starts again.
That is the behaviour a test-day forecast needs.

**Next:** once the platform has enough of its own review logs, refit FSRS's parameters on them (py-fsrs ships an
optimizer) and compare against these defaults on held-out reviews, the evaluation this notebook could not do
with Duolingo data.